In [11]:

!pip install -q faiss-gpu rank-bm25 pdfplumber python-docx langchain-text-splitters fastapi uvicorn pyngrok nest-asyncio llama-index

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.2/135.2 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 128.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 127.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 127.3 MB/s eta 0:00:00


In [15]:
!pip uninstall -y Pillow pillow
!pip install Pillow==11.3.0 --no-cache-dir

Found existing installation: pillow 12.3.0
Uninstalling pillow-12.3.0:
  Successfully uninstalled pillow-12.3.0
  Using cached pillow-12.3.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
Using cached pillow-12.3.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (6.9 MB)


In [1]:
from sentence_transformers import SentenceTransformer


In [11]:
!pip install -U llama-index-readers-file docx2txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 114.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.9/164.9 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 17.8 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: nltk
    Found existing installation: nltk 3.9.1
    Uninstalling nltk-3.9.1:
      Successfully uninstalled nltk-3.9.1
ERROR: pip's dependency resolver does not curre

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import re
import zipfile
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
from urllib.parse import urljoin

In [3]:
DOCS_DIR = "docs"

SPECS = {
    "38.300": "https://www.3gpp.org/ftp/Specs/archive/38_series/38.300/",
    "23.501": "https://www.3gpp.org/ftp/Specs/archive/23_series/23.501/",
    "33.501": "https://www.3gpp.org/ftp/Specs/archive/33_series/33.501/",
}

In [4]:
TODAY = datetime.now()
START_DATE = TODAY - timedelta(days=365)

In [5]:
def get_directory_files(url):
    """Read the 3GPP directory and return ZIP files with dates."""

    print(f"\nChecking: {url}")

    response = requests.get(
        url,
        timeout=30,
        headers={
            "User-Agent": "Mozilla/5.0"
        }
    )

    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    files = []

    for row in soup.find_all("tr"):

        link = row.find("a")

        if not link:
            continue

        href = link.get("href", "")
        name = link.text.strip()

        if not name.lower().endswith(".zip"):
            continue

        # Extract date from row
        text = row.get_text(" ", strip=True)

        # Example:
        # 38300-j10.zip 2026/01/14 23:43 7708,5 KB
        match = re.search(
            r"(\d{4}/\d{2}/\d{2})",
            text
        )

        if not match:
            continue

        date_str = match.group(1)

        try:
            file_date = datetime.strptime(
                date_str,
                "%Y/%m/%d"
            )
        except ValueError:
            continue

        files.append({
            "name": name,
            "url": urljoin(url, href),
            "date": file_date
        })

    return files

In [6]:
def download_file(url, path):
    """Download a file with progress."""

    if os.path.exists(path):
        print(f"Already exists: {path}")
        return

    print(f"Downloading: {os.path.basename(path)}")

    response = requests.get(
        url,
        stream=True,
        timeout=120,
        headers={
            "User-Agent": "Mozilla/5.0"
        }
    )

    response.raise_for_status()

    total = int(
        response.headers.get("content-length", 0)
    )

    downloaded = 0

    with open(path, "wb") as f:

        for chunk in response.iter_content(
            chunk_size=1024 * 1024
        ):

            if not chunk:
                continue

            f.write(chunk)
            downloaded += len(chunk)

            if total:
                percent = downloaded * 100 / total
                print(
                    f"\r  {percent:6.2f}%",
                    end=""
                )

    print()

In [7]:
def extract_documents(zip_path, output_dir):

    DOCUMENT_EXTENSIONS = {
        ".pdf",
        ".doc",
        ".docx",
        ".txt",
        ".md",
        ".html",
        ".htm",
        ".xml",
        ".rtf",
        ".odt",
        ".csv",
    }

    os.makedirs(output_dir, exist_ok=True)

    print(f"Extracting: {os.path.basename(zip_path)}")

    extracted = 0

    with zipfile.ZipFile(zip_path, "r") as z:

        for member in z.infolist():

            if member.is_dir():
                continue

            filename = member.filename
            ext = os.path.splitext(filename)[1].lower()

            if ext not in DOCUMENT_EXTENSIONS:
                continue

            basename = os.path.basename(filename)

            if not basename:
                continue

            output_path = os.path.join(
                output_dir,
                basename
            )

            # Prevent overwriting
            if os.path.exists(output_path):

                name, ext = os.path.splitext(basename)
                counter = 1

                while os.path.exists(output_path):

                    new_name = f"{name}_{counter}{ext}"

                    output_path = os.path.join(
                        output_dir,
                        new_name
                    )

                    counter += 1

            with z.open(member) as source:
                with open(output_path, "wb") as target:
                    target.write(source.read())

            print(f"  -> {os.path.basename(output_path)}")

            extracted += 1

    print(f"Extracted {extracted} document files.")

    return extracted

In [8]:
def main():

    os.makedirs(
        DOCS_DIR,
        exist_ok=True
    )

    print("=" * 70)
    print("3GPP DOCUMENT DOWNLOADER")
    print("=" * 70)

    print(
        f"Today      : {TODAY.date()}"
    )

    print(
        f"From       : {START_DATE.date()}"
    )

    print(
        f"Output     : "
        f"{os.path.abspath(DOCS_DIR)}"
    )

    total_archives = 0
    total_documents = 0

    # ========================================================
    # PROCESS EACH SPECIFICATION
    # ========================================================

    for spec, url in SPECS.items():

        print("\n")
        print("=" * 70)
        print(f"TS {spec}")
        print("=" * 70)

        try:

            files = get_directory_files(url)

        except Exception as e:

            print(
                f"ERROR reading directory: {e}"
            )

            continue

        # ----------------------------------------------------
        # Filter last year
        # ----------------------------------------------------

        recent_files = [
            f
            for f in files
            if START_DATE <= f["date"] <= TODAY
        ]

        recent_files.sort(
            key=lambda x: x["date"]
        )

        print(
            f"\nFound "
            f"{len(recent_files)} "
            f"revisions."
        )

        # ----------------------------------------------------
        # Specification folder
        # ----------------------------------------------------

        spec_dir = os.path.join(
            DOCS_DIR,
            spec.replace(".", "_")
        )

        os.makedirs(
            spec_dir,
            exist_ok=True
        )

        # ----------------------------------------------------
        # Download + extract
        # ----------------------------------------------------

        for file in recent_files:

            print("\n" + "-" * 70)

            print(
                f"Revision : {file['name']}"
            )

            print(
                f"Date     : "
                f"{file['date'].date()}"
            )

            zip_path = os.path.join(
                spec_dir,
                file["name"]
            )

            try:

                download_file(
                    file["url"],
                    zip_path
                )

                extracted = extract_documents(
                    zip_path,
                    spec_dir
                )

                total_archives += 1
                total_documents += extracted

                # Remove ZIP after extraction
                os.remove(zip_path)

                print(
                    "ZIP removed."
                )

            except Exception as e:

                print(
                    f"ERROR: {e}"
                )

    # ========================================================
    # SUMMARY
    # ========================================================

    print("\n")
    print("=" * 70)
    print("DOWNLOAD COMPLETE")
    print("=" * 70)

    print(
        f"Archives processed : "
        f"{total_archives}"
    )

    print(
        f"Documents extracted: "
        f"{total_documents}"
    )

    print(
        f"Location            : "
        f"{os.path.abspath(DOCS_DIR)}"
    )


In [10]:
main()

3GPP DOCUMENT DOWNLOADER
Today      : 2026-08-15
From       : 2025-08-15
Output     : /content/docs


TS 38.300

Checking: https://www.3gpp.org/ftp/Specs/archive/38_series/38.300/

Found 16 revisions.

----------------------------------------------------------------------
Revision : 38300-i70.zip
Date     : 2025-10-01
Downloading: 38300-i70.zip
  100.00%
Extracting: 38300-i70.zip
  -> 38300-i70.docx
Extracted 1 document files.
ZIP removed.

----------------------------------------------------------------------
Revision : 38300-j00.zip
Date     : 2025-10-03
Downloading: 38300-j00.zip
  100.00%
Extracting: 38300-j00.zip
  -> 38300-j00.docx
Extracted 1 document files.
ZIP removed.

----------------------------------------------------------------------
Revision : 38300-he0.zip
Date     : 2025-10-10
Downloading: 38300-he0.zip
  100.00%
Extracting: 38300-he0.zip
  -> 38300-he0.docx
Extracted 1 document files.
ZIP removed.

---------------------------------------------------------------------

In [12]:
from sentence_transformers import SentenceTransformer

In [9]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import HierarchicalNodeParser
from llama_index.core.ingestion import IngestionPipeline

# Load all supported documents recursively
documents = SimpleDirectoryReader(
    input_dir=DOCS_DIR,
    recursive=True,
).load_data()

print(f"Loaded documents: {len(documents)}")

# Hierarchical parser
node_parser = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[2048, 512, 128],
    chunk_overlap=20,
)

# Create ingestion pipeline
pipeline = IngestionPipeline(
    transformations=[
        node_parser
    ]
)

# Parse documents into hierarchical nodes
nodes = pipeline.run(
    documents=documents
)

print(f"Total nodes: {len(nodes)}")

Loaded documents: 41
Total nodes: 389228


In [14]:
import torch
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [15]:
from sentence_transformers import SentenceTransformer
import torch
import faiss
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)

# Extract text from nodes
texts = [node.text for node in nodes]

# Embed in batches (GPU)
embeddings = embed_model.encode(texts, show_progress_bar=True, batch_size=64)

# Build FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(np.array(embeddings).astype('float32'))

# Save for later
import pickle
with open('nodes.pkl', 'wb') as f:
    pickle.dump(nodes, f)
faiss.write_index(index, 'faiss.index')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/6082 [00:00<?, ?it/s]

In [40]:
import pickle

with open("/content/drive/MyDrive/nodes.pkl", "rb") as f:
    nodes = pickle.load(f)

texts = [node.text for node in nodes]

with open("/content/drive/MyDrive/texts.pkl", "wb") as f:
    pickle.dump(texts, f)

print(f"✅ Created texts.pkl with {len(texts)} text chunks.")

/usr/local/lib/python3.12/dist-packages/pydantic/main.py:253: RuntimeWarning: coroutine 'Server.serve' was never awaited
  def __init__(self, /, **data: Any) -> None:


✅ Created texts.pkl with 389228 text chunks.
